In [2]:
import anndata as ad
import pandas as pd
import numpy as np

# Get Pubchem

In [3]:
op3 = ad.read_h5ad('../../../data/op3/pseudobulk_processed/sep_rep/op3_standardized_processed.h5ad')
op3_obs = op3.obs
df_op3_sm = op3_obs.drop_duplicates(['perturbagen', 'pubchem_cid'])[['perturbagen', 'pubchem_cid']].reset_index(drop=True)

In [4]:
df_op3_sm_add = pd.DataFrame({'perturbagen': ['Belinostat', 'Dabrafenib'],
                              'pubchem_cid': [6918638, 44462760]})

In [5]:
df_emb_op3 = pd.concat([df_op3_sm, df_op3_sm_add]).reset_index(drop=True)

In [6]:
df_emb_op3

,perturbagen,pubchem_cid
0,TIE2 Kinase Inhibitor,23625762
1,MK-5108,24748204
2,Lapatinib,208908
3,Dimethyl Sulfoxide,679
4,Atorvastatin,60823
...,...,...
136,Vanoxerine,3455
137,SB525334,9967941
138,HYDROXYUREA,3657
139,Belinostat,6918638


In [7]:
df_emb_op3['pubchem_cid'] = df_emb_op3['pubchem_cid'].astype(str)

# Get embeddings

In [8]:
df_pert = pd.read_pickle("../../../lpm_style/lpm_style_embeddings_epoch_5/df_pert.pkl")

In [9]:
df_emb_op3 = df_emb_op3.merge(df_pert, left_on='pubchem_cid', right_on='symbol')

In [10]:
df_emb_op3

,perturbagen,pubchem_cid,symbol,code,lpm_style_embeddings
0,TIE2 Kinase Inhibitor,23625762,23625762,8274,"[0.23538774251937866, 0.0699068158864975, 0.04..."
1,MK-5108,24748204,24748204,8813,"[0.12560248374938965, -0.008512352593243122, -..."
2,Lapatinib,208908,208908,7670,"[0.002801814815029502, 0.31994032859802246, 0...."
3,Atorvastatin,60823,60823,31249,"[-0.2172117382287979, -0.15038855373859406, 0...."
4,Ganetespib (STA-9090),135564985,135564985,4124,"[0.07162753492593765, 0.2915172576904297, 0.06..."
...,...,...,...,...,...
133,Vanoxerine,3455,3455,10855,"[-0.038833875209093094, -0.0400872603058815, -..."
134,SB525334,9967941,9967941,37412,"[-0.26656079292297363, -0.12373765558004379, 0..."
135,HYDROXYUREA,3657,3657,11040,"[-0.1603565812110901, 0.3974446952342987, 0.16..."
136,Belinostat,6918638,6918638,32929,"[-0.009625555947422981, -0.04900122433900833, ..."


# Get FP

In [11]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

In [12]:
de_train = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')
de_test = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad')

In [13]:
pd.read_pickle("../../data_mol_emb_split/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb.pkl")

,perturbagen,LPM_emb,smiles,ECFP:2
0,TIE2 Kinase Inhibitor,"[0.16717687249183655, -0.20145422220230103, 0....",COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,MK-5108,"[-0.144333153963089, 0.06810502707958221, -0.0...",O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Lapatinib,"[-0.06751061975955963, -0.12120415270328522, 0...",CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Belinostat,"[-0.0318511500954628, -0.16489124298095703, 0....",O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Dabrafenib,"[0.09784593433141708, -0.4855876863002777, -0....",CC(C)(C)c1nc(-c2cccc(NS(=O)(=O)c3c(F)cccc3F)c2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
135,GLPG0634,"[0.016147663816809654, -0.14773432910442352, -...",O=C(Nc1nc2cccc(-c3ccc(CN4CCS(=O)(=O)CC4)cc3)n2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
136,Mubritinib (TAK 165),"[0.07429785281419754, -0.01757134683430195, 0....",FC(F)(F)c1ccc(/C=C/c2nc(COc3ccc(CCCCn4ccnn4)cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
137,Vanoxerine,"[0.08700791746377945, 0.2101263850927353, -0.0...",Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
138,SB525334,"[-0.0031143922824412584, 0.11767911165952682, ...",Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [14]:
de = ad.concat([de_train, de_test])

In [15]:
sm_smiles = de.obs[['sm_name', 'SMILES']].drop_duplicates()

In [16]:
sm_smiles['ECFP:2'] = smiles_to_fingerprints(sm_smiles['SMILES'])

In [17]:
sm_smiles = sm_smiles.rename(columns = {'SMILES': 'smiles', 'sm_name': 'perturbagen'})

In [18]:
sm_smiles

,perturbagen,smiles,ECFP:2
"NK cells, TIE2 Kinase Inhibitor",TIE2 Kinase Inhibitor,COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, MK-5108",MK-5108,O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Lapatinib",Lapatinib,CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, Belinostat",Belinostat,O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, Dabrafenib",Dabrafenib,CC(C)(C)c1nc(-c2cccc(NS(=O)(=O)c3c(F)cccc3F)c2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...
"B cells, GLPG0634",GLPG0634,O=C(Nc1nc2cccc(-c3ccc(CN4CCS(=O)(=O)CC4)cc3)n2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Mubritinib (TAK 165)",Mubritinib (TAK 165),FC(F)(F)c1ccc(/C=C/c2nc(COc3ccc(CCCCn4ccnn4)cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Vanoxerine",Vanoxerine,Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, SB525334",SB525334,Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


# JOIN

In [19]:
df_emb_op3_merged = df_emb_op3.merge(sm_smiles, how='left', left_on='perturbagen', right_on='perturbagen').rename(columns={'lpm_style_embeddings': 'LPM_emb'})[['perturbagen', 'LPM_emb', 'smiles', 'ECFP:2']].reset_index(drop=True)
df_emb_op3_merged#.to_pickle('../../data/benchmark/resources/datasets/neurips-2023-data-subsample-lpm-style/op3_emb.pkl')

,perturbagen,LPM_emb,smiles,ECFP:2
0,TIE2 Kinase Inhibitor,"[0.23538774251937866, 0.0699068158864975, 0.04...",COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,MK-5108,"[0.12560248374938965, -0.008512352593243122, -...",O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Lapatinib,"[0.002801814815029502, 0.31994032859802246, 0....",CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Atorvastatin,"[-0.2172117382287979, -0.15038855373859406, 0....",CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Ganetespib (STA-9090),"[0.07162753492593765, 0.2915172576904297, 0.06...",CC(C)c1cc(-c2n[nH]c(=O)n2-c2ccc3c(ccn3C)c2)c(O...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
133,Vanoxerine,"[-0.038833875209093094, -0.0400872603058815, -...",Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
134,SB525334,"[-0.26656079292297363, -0.12373765558004379, 0...",Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
135,HYDROXYUREA,"[-0.1603565812110901, 0.3974446952342987, 0.16...",NC(O)=NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
136,Belinostat,"[-0.009625555947422981, -0.04900122433900833, ...",O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [26]:
import os
os.makedirs('../../data/benchmark/resources/datasets/neurips-2023-data-subsample-lpm-style')
df_emb_op3_merged.to_pickle('../../data/benchmark/resources/datasets/neurips-2023-data-subsample-lpm-style/op3_emb.pkl')

In [27]:
df_emb_op3_merged

,perturbagen,LPM_emb,smiles,ECFP:2
0,TIE2 Kinase Inhibitor,"[0.23538774251937866, 0.0699068158864975, 0.04...",COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,MK-5108,"[0.12560248374938965, -0.008512352593243122, -...",O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Lapatinib,"[0.002801814815029502, 0.31994032859802246, 0....",CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Atorvastatin,"[-0.2172117382287979, -0.15038855373859406, 0....",CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Ganetespib (STA-9090),"[0.07162753492593765, 0.2915172576904297, 0.06...",CC(C)c1cc(-c2n[nH]c(=O)n2-c2ccc3c(ccn3C)c2)c(O...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
133,Vanoxerine,"[-0.038833875209093094, -0.0400872603058815, -...",Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
134,SB525334,"[-0.26656079292297363, -0.12373765558004379, 0...",Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
135,HYDROXYUREA,"[-0.1603565812110901, 0.3974446952342987, 0.16...",NC(O)=NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
136,Belinostat,"[-0.009625555947422981, -0.04900122433900833, ...",O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


# SPLIT

In [28]:
import anndata as ad
import pandas as pd
import pickle

In [29]:
import numpy as np

In [30]:
ratio = 0.25

In [31]:
op3_train_subsample = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')
op3_test_subsample = ad.read_h5ad('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad')
df = pd.read_csv('../../data_mol_emb/benchmark/resources/datasets/neurips-2023-data/id_map.csv')

In [32]:
path = '../../data/benchmark/resources/datasets/neurips-2023-data-subsample-lpm-style/op3_emb.pkl'
with open(path, 'rb') as fp:
    op3_emb = pickle.load(fp)

In [33]:
op3_train_subsample = op3_train_subsample[op3_train_subsample.obs['sm_name'].isin(op3_emb['perturbagen'])].copy()
op3_test_subsample = op3_test_subsample[op3_test_subsample.obs['sm_name'].isin(op3_emb['perturbagen'])].copy()

In [34]:
op3_test_subsample

AnnData object with n_obs × n_vars = 149 × 5317
    obs: 'sm_cell_type', 'cell_type', 'sm_name', 'sm_lincs_id', 'SMILES', 'split', 'control'
    uns: 'dataset_description', 'dataset_id', 'dataset_name', 'dataset_organism', 'dataset_reference', 'dataset_summary', 'dataset_url', 'single_cell_obs'
    layers: 'AveExpr', 'B', 'P.Value', 'adj.P.Value', 'clipped_sign_log10_pval', 'is_de', 'is_de_adj', 'logFC', 'sign_log10_adj_pval', 'sign_log10_pval', 't'

In [35]:
op3_subsample = ad.concat([op3_train_subsample, op3_test_subsample], uns_merge='same')

In [36]:
df_single_cell_obs = pd.concat([op3_train_subsample.uns['single_cell_obs'], op3_test_subsample.uns['single_cell_obs']])

In [37]:
compounds = np.array(op3_subsample.obs['sm_name'].unique())

In [38]:
np.random.seed(42)
test_sample = np.random.choice(compounds, size=int(len(compounds) * ratio), replace=False)

In [39]:
op3_subsample.obs['new_split'] = np.where(op3_subsample.obs['sm_name'].isin(test_sample), 'test', 'train')

In [40]:
op3_train_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'train'].copy()
op3_train_subsample_.uns['single_cell_obs'] = df_single_cell_obs[~df_single_cell_obs['sm_name'].isin(test_sample)]
op3_train_subsample_.write_h5ad('../../data/benchmark/resources/datasets/neurips-2023-data-subsample-lpm-style/de_train.h5ad', compression='gzip')

In [41]:
op3_test_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'test'].copy()
op3_test_subsample_.uns['single_cell_obs'] = df_single_cell_obs[df_single_cell_obs['sm_name'].isin(test_sample)]
op3_test_subsample_.write_h5ad('../../data/benchmark/resources/datasets/neurips-2023-data-subsample-lpm-style/de_test.h5ad', compression='gzip')

In [42]:
op3_test_subsample_.obs[['sm_name', 'cell_type']].reset_index(drop=True).reset_index().rename(columns={'index': 'id'}).to_csv('../../data/benchmark/resources/datasets/neurips-2023-data-subsample-lpm-style/id_map.csv', index=False)

# Check emb

In [43]:
op3_train_subsample_

AnnData object with n_obs × n_vars = 411 × 5317
    obs: 'sm_cell_type', 'cell_type', 'sm_name', 'sm_lincs_id', 'SMILES', 'split', 'control', 'new_split'
    uns: 'dataset_description', 'dataset_id', 'dataset_name', 'dataset_organism', 'dataset_reference', 'dataset_summary', 'dataset_url', 'single_cell_obs'
    layers: 'AveExpr', 'B', 'P.Value', 'adj.P.Value', 'clipped_sign_log10_pval', 'is_de', 'is_de_adj', 'logFC', 'sign_log10_adj_pval', 'sign_log10_pval', 't'

In [46]:
adata_train_prev = ad.read_h5ad('../../data_lpm_stype_epoch1/benchmark/resources/datasets/neurips-2023-data-subsample/de_train.h5ad')

In [47]:
adata_test_prev = ad.read_h5ad('../../data_lpm_stype_epoch1/benchmark/resources/datasets/neurips-2023-data-subsample/de_test.h5ad')

In [48]:
op3_pickle = pd.read_pickle('../../data_lpm_stype_epoch1/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb.pkl')

In [55]:
(op3_test_subsample_.layers['clipped_sign_log10_pval'] == adata_test_prev.layers['clipped_sign_log10_pval']).all()

True

In [56]:
op3_pickle

,perturbagen,LPM_emb,smiles,ECFP:2
0,TIE2 Kinase Inhibitor,"[-0.02075847052037716, 0.19725458323955536, -0...",COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,MK-5108,"[-0.10335095226764679, 0.3252302408218384, -0....",O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Lapatinib,"[-0.027052316814661026, 0.13099196553230286, 0...",CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Atorvastatin,"[0.11225434392690659, -0.0793108344078064, 0.0...",CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Ganetespib (STA-9090),"[-0.0544951967895031, -0.3173687160015106, -0....",CC(C)c1cc(-c2n[nH]c(=O)n2-c2ccc3c(ccn3C)c2)c(O...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
133,Vanoxerine,"[0.03207073733210564, 0.1282859891653061, -0.0...",Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
134,SB525334,"[0.45691099762916565, 0.18649746477603912, -0....",Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
135,HYDROXYUREA,"[-0.07913605123758316, -0.0068639772944152355,...",NC(O)=NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
136,Belinostat,"[-0.18465366959571838, 0.06446841359138489, -0...",O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [57]:
op3_emb

,perturbagen,LPM_emb,smiles,ECFP:2
0,TIE2 Kinase Inhibitor,"[0.23538774251937866, 0.0699068158864975, 0.04...",COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,MK-5108,"[0.12560248374938965, -0.008512352593243122, -...",O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Lapatinib,"[0.002801814815029502, 0.31994032859802246, 0....",CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Atorvastatin,"[-0.2172117382287979, -0.15038855373859406, 0....",CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Ganetespib (STA-9090),"[0.07162753492593765, 0.2915172576904297, 0.06...",CC(C)c1cc(-c2n[nH]c(=O)n2-c2ccc3c(ccn3C)c2)c(O...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
133,Vanoxerine,"[-0.038833875209093094, -0.0400872603058815, -...",Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
134,SB525334,"[-0.26656079292297363, -0.12373765558004379, 0...",Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
135,HYDROXYUREA,"[-0.1603565812110901, 0.3974446952342987, 0.16...",NC(O)=NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
136,Belinostat,"[-0.009625555947422981, -0.04900122433900833, ...",O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
